# Objetivo

Na etapa de modelagem estatística, verificou-se que o problema apresenta relações não lineares e possíveis interações entre segmentos de clientes que não foram plenamente representadas pela Regressão Logística. Embora o modelo tenha alcançado acurácia aproximada de 84%, apresentou baixo recall para a classe churn, deixando de identificar uma parcela relevante dos clientes que efetivamente abandonaram o banco.

Diante disso, este notebook tem como objetivo avaliar algoritmos de Machine Learning capazes de representar relações não lineares e melhorar a identificação da classe churn. A comparação dará atenção especial ao recall, ao F1-score e à PR-AUC, buscando modelos que apresentem melhor capacidade de discriminar clientes em risco sem produzir quantidade excessiva de falsos positivos.

Na etapa de otimização, os hiperparâmetros serão selecionados com base na PR-AUC, por se tratar de uma métrica independente de um único limiar de classificação e adequada à avaliação da classe minoritária. Posteriormente, o limiar de decisão poderá ser ajustado para privilegiar o recall e reduzir o número de falsos negativos.

# Preparação

In [1]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve, fbeta_score, make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

In [2]:
df = pd.read_csv('../data/raw/Churn_Modelling.csv', sep=',', na_values=[''], quotechar='"')

Remoção dos atributos que apresentam identificadores e não representam atributo comportamentais dos clientes.

In [3]:
df.drop(columns=['CustomerId', 'RowNumber', 'Surname'], inplace=True)

In [4]:
y = df["Exited"]

Tratamento do Complete Separation: clientes com 3+ produtos apresentam comportamento bastante diferentes do restante do conjunto de dados

In [5]:
df["ProductsGroup"] = df["NumOfProducts"].replace({
    1: "1",
    2: "2",
    3: "3+",
    4: "3+",
})

df["ProductsGroup"] = pd.Categorical(df["ProductsGroup"], categories=["1", "2", "3+"])

X = df.drop(columns=["Exited"])

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [7]:

baseline_models = {
    "Regressão Logística": LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
}

machine_learning_models = {
    "KNN": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    ),

    "SVM RBF": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight=None,
        random_state=RANDOM_STATE,
    ),

    "Decision Tree": DecisionTreeClassifier(
        criterion="gini",
        max_depth=6,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight=None,
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        criterion="gini",
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        bootstrap=True,
        class_weight=None,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=RANDOM_STATE,
    ),
}

In [8]:

numeric_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
]

categorical_features = [
    "Geography",
    "Gender",
    "ProductsGroup",
]

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# Comparação dos modelos por validação cruzada estratificada


In [9]:
scaled_model_names = {
    "Regressão Logística",
    "KNN",
    "SVM RBF",
}

all_models = {
    **baseline_models,
    **machine_learning_models,
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

resultados_cv = []

for model_name, estimator in all_models.items():
    preprocessor = (
        preprocessor_scaled
        if model_name in scaled_model_names
        else preprocessor_tree
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train,
        Y_train,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    )

    resultados_cv.append({
        "Modelo": model_name,
        "Accuracy média": scores["test_accuracy"].mean(),
        "Balanced Accuracy média":
            scores["test_balanced_accuracy"].mean(),
        "Precision média": scores["test_precision"].mean(),
        "Recall médio": scores["test_recall"].mean(),
        "F1 médio": scores["test_f1"].mean(),
        "ROC-AUC médio": scores["test_roc_auc"].mean(),
        "PR-AUC médio": scores["test_pr_auc"].mean(),
        "Recall desvio-padrão": scores["test_recall"].std(),
        "F1 desvio-padrão": scores["test_f1"].std(),
        "PR-AUC desvio-padrão": scores["test_pr_auc"].std(),
    })

resultados_df = (
    pd.DataFrame(resultados_cv)
    .set_index("Modelo")
    .sort_values("Recall médio", ascending=False)
)

resultados_df

/home/rodrigo-emygdio/Estudos/Faculdade/PUC/IA-MACHINE-LEARNING/Projetos_Finais/ML_E_Modelagem_Preditiva/bank-churn-analytics/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/rodrigo-emygdio/Estudos/Faculdade/PUC/IA-MACHINE-LEARNING/Projetos_Finais/ML_E_Modelagem_Preditiva/bank-churn-analytics/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of pe

,Accuracy média,Balanced Accuracy média,Precision média,Recall médio,F1 médio,ROC-AUC médio,PR-AUC médio,Recall desvio-padrão,F1 desvio-padrão,PR-AUC desvio-padrão
Modelo,,,,,,,,,,
Gradient Boosting,0.863500,0.717301,0.769136,0.470552,0.583541,0.865549,0.696938,0.037297,0.036425,0.015842
Random Forest,0.861625,0.709048,0.775080,0.451534,0.570511,0.858446,0.683856,0.023232,0.023341,0.015734
Decision Tree,0.856375,0.699132,0.757847,0.433742,0.551609,0.835465,0.632119,0.020030,0.022597,0.018476
SVM RBF,0.854250,0.675885,0.805313,0.374847,0.511176,0.826011,0.657694,0.029062,0.029649,0.026549
Regressão Logística,0.840000,0.666024,0.702162,0.372393,0.486479,0.831286,0.614766,0.021481,0.020838,0.016904
KNN,0.836250,0.640843,0.730884,0.311043,0.436319,0.813549,0.588226,0.016301,0.022687,0.043604


A validação cruzada estratificada confirmou o Gradient Boosting como o modelo de melhor desempenho global, apresentando os maiores valores médios de recall, F1-score, ROC-AUC e PR-AUC. O Random Forest apresentou desempenho próximo e maior estabilidade no recall. Esses resultados justificam a seleção dos modelos de ensemble para a etapa de otimização de hiperparâmetros.

Os resultados obtidos por validação cruzada mostraram comportamento consistente entre os diferentes algoritmos avaliados. O Gradient Boosting apresentou o melhor desempenho em todas as principais métricas relacionadas à identificação da classe minoritária, alcançando o maior recall médio (47%), F1-score (0,584) e PR-AUC (0,697).

O Random Forest apresentou desempenho muito próximo, confirmando a capacidade dos métodos baseados em árvores de capturar relações não lineares e interações entre variáveis identificadas durante a etapa de modelagem estatística. Em contraste, modelos lineares, como a Regressão Logística, e métodos baseados em distância, como o KNN, apresentaram desempenho inferior na identificação da classe de churn.

O SVM apesar de recall médio inferior, teve um PR-AUC superior ao da Decision Tree e apresenta uma capacidade de ranqueamento da classe churn melhor.

Esses resultados corroboram os achados obtidos durante a análise estatística, que indicaram a presença de relações não lineares, segmentos específicos de clientes e padrões de interação não explicitamente modelados pela regressão logística. Os modelos baseados em árvores mostraram-se mais adequados para representar essa estrutura dos dados.

Embora o Gradient Boosting tenha apresentado o melhor desempenho, o recall médio permaneceu inferior a 50%, indicando que uma parcela considerável dos clientes que realizam churn continua não sendo identificada. Esse comportamento é consistente com a análise dos resíduos realizada anteriormente, sugerindo que parte desse erro decorre da ausência de variáveis comportamentais relevantes no conjunto de dados, e não apenas da escolha do algoritmo.

# Seleção dos modelos para otimização

Foram selecionados para otimização o Gradient Boosting e o Random Forest, por apresentarem os melhores resultados globais; a Decision Tree, por alcançar o terceiro maior recall médio e permitir avaliar isoladamente o comportamento da estrutura que fundamenta os modelos de ensemble; e o SVM RBF, por apresentar PR-AUC superior à árvore individual, apesar do menor recall no limiar padrão. Dessa forma, a seleção considerou tanto o desempenho observado quanto a diversidade de funcionamento dos algoritmos.

In [14]:
models_for_optimization = {
    "Gradient Boosting": Pipeline(
        steps=[
            ("preprocessor", preprocessor_tree),
            (
                "model",
                GradientBoostingClassifier(
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor_tree),
            (
                "model",
                RandomForestClassifier(
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),

    "Decision Tree": Pipeline(
        steps=[
            ("preprocessor", preprocessor_tree),
            (
                "model",
                DecisionTreeClassifier(
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "SVM RBF": Pipeline(
        steps=[
            ("preprocessor", preprocessor_scaled),
            (
                "model",
                SVC(
                    kernel="rbf",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

## Otimização com GridSearchCV

Busca de hiperparâmetros nos modelos mais promissores para verificar se há ganho real em relação ao baseline.

In [15]:
param_grids = {
    "Gradient Boosting": {
        "model__n_estimators": [100, 150, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
        "model__min_samples_leaf": [5, 10],
        "model__subsample": [0.8, 1.0],
    },

    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [None, 8],
        "model__min_samples_split": [5, 10],
        "model__min_samples_leaf": [2, 4],
        "model__max_features": ["sqrt"],
        "model__class_weight": [None, "balanced"],
    },

    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [3, 5, 7, None],
        "model__min_samples_split": [10, 20, 40],
        "model__min_samples_leaf": [5, 10, 20],
        "model__class_weight": [None, "balanced"],
    },

    "SVM RBF": {
        "model__C": [0.5, 1.0, 2.0],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__class_weight": [None, "balanced"],
    },
}

scoring_grid_search = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

refit="pr_auc"



In [16]:
grid_searches = {}
best_models = {}
grid_search_summary = []
grid_search_results = {}

for model_name, pipeline in models_for_optimization.items():

    print(f"Executando GridSearchCV para: {model_name}")

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring=scoring,
        refit=refit,
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
    )

    grid_search.fit(X_train, Y_train)

    # Objeto completo do GridSearch
    grid_searches[model_name] = grid_search

    # Melhor pipeline, já treinado em todo o conjunto de treino
    best_models[model_name] = grid_search.best_estimator_

    # Resultados completos de todas as combinações
    grid_search_results[model_name] = pd.DataFrame(
        grid_search.cv_results_
    )

    # Índice da melhor configuração segundo a métrica de refit
    best_index = grid_search.best_index_

    # Resumo da melhor configuração
    grid_search_summary.append({
        "Model": model_name,
        "Best Parameters": grid_search.best_params_,

        "Train PR-AUC": (
            grid_search.cv_results_["mean_train_pr_auc"][best_index]
        ),
        "CV PR-AUC": (
            grid_search.cv_results_["mean_test_pr_auc"][best_index]
        ),
        "CV PR-AUC Std": (
            grid_search.cv_results_["std_test_pr_auc"][best_index]
        ),

        "CV Recall": (
            grid_search.cv_results_["mean_test_recall"][best_index]
        ),
        "CV Recall Std": (
            grid_search.cv_results_["std_test_recall"][best_index]
        ),

        "CV Precision": (
            grid_search.cv_results_["mean_test_precision"][best_index]
        ),

        "CV F1": (
            grid_search.cv_results_["mean_test_f1"][best_index]
        ),

        "CV ROC-AUC": (
            grid_search.cv_results_["mean_test_roc_auc"][best_index]
        ),

        "CV Balanced Accuracy": (
            grid_search.cv_results_[
                "mean_test_balanced_accuracy"
            ][best_index]
        ),

        "CV Accuracy": (
            grid_search.cv_results_["mean_test_accuracy"][best_index]
        ),

        "PR-AUC Train-CV Gap": (
            grid_search.cv_results_["mean_train_pr_auc"][best_index]
            - grid_search.cv_results_["mean_test_pr_auc"][best_index]
        ),
    })

    print(
        f"Melhor PR-AUC: {grid_search.best_score_:.4f}"
    )
    print(
        f"Melhores parâmetros: {grid_search.best_params_}"
    )
    print("-" * 80)

Executando GridSearchCV para: Gradient Boosting
Melhor PR-AUC: 0.6988
Melhores parâmetros: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_samples_leaf': 10, 'model__n_estimators': 200, 'model__subsample': 0.8}
--------------------------------------------------------------------------------
Executando GridSearchCV para: Random Forest
Melhor PR-AUC: 0.6856
Melhores parâmetros: {'model__class_weight': None, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 10, 'model__n_estimators': 300}
--------------------------------------------------------------------------------
Executando GridSearchCV para: Decision Tree
Melhor PR-AUC: 0.6443
Melhores parâmetros: {'model__class_weight': 'balanced', 'model__criterion': 'entropy', 'model__max_depth': 7, 'model__min_samples_leaf': 20, 'model__min_samples_split': 10}
--------------------------------------------------------------------------------
Executando GridSearchCV par

In [18]:
grid_search_summary_df = (
    pd.DataFrame(grid_search_summary)
    .sort_values(
        by="CV PR-AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)
grid_search_summary_df

,Model,Best Parameters,Train PR-AUC,CV PR-AUC,CV PR-AUC Std,CV Recall,CV Recall Std,CV Precision,CV F1,CV ROC-AUC,CV Balanced Accuracy,CV Accuracy,PR-AUC Train-CV Gap
0,Gradient Boosting,"{'model__learning_rate': 0.05, 'model__max_dep...",0.756351,0.698766,0.016294,0.476074,0.034835,0.769112,0.587768,0.865916,0.719826,0.864250,0.057586
1,Random Forest,"{'model__class_weight': None, 'model__max_dept...",0.783105,0.685624,0.013156,0.414110,0.027505,0.800157,0.545465,0.859410,0.693868,0.859625,0.097481
2,SVM RBF,"{'model__C': 0.5, 'model__class_weight': None,...",0.713817,0.659284,0.025689,0.362577,0.026345,0.817366,0.501846,0.828305,0.670927,0.853625,0.054533
3,Decision Tree,"{'model__class_weight': 'balanced', 'model__cr...",0.708725,0.644293,0.025141,0.733129,0.039139,0.473499,0.574299,0.835367,0.761541,0.778375,0.064432


Os resultados evidenciam diferentes perfis de decisão entre os modelos otimizados. Gradient Boosting, Random Forest e SVM apresentaram comportamento mais conservador, caracterizado por maior precisão e menor recall. Em contraste, a Decision Tree, cuja melhor configuração utilizou ponderação balanceada das classes, alcançou recall substancialmente superior, identificando aproximadamente 73% dos casos reais de churn. Esse ganho ocorreu à custa de uma redução da precisão para aproximadamente 47%, indicando maior incidência de falsos positivos. Assim, a árvore de decisão ocupou uma região distinta do compromisso entre precisão e recall, tornando-se potencialmente adequada para cenários em que o custo de não identificar um cliente propenso ao churn seja superior ao custo de uma intervenção desnecessária.

# Avaliação final no conjunto de teste

In [19]:
test_results = []
test_predictions = {}
test_scores = {}

for model_name, model in best_models.items():

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    test_predictions[model_name] = y_pred
    test_scores[model_name] = y_score

    tn, fp, fn, tp = confusion_matrix(Y_test, y_pred).ravel()

    test_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(Y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(Y_test, y_pred),
        "Precision": precision_score(
            Y_test,
            y_pred,
            zero_division=0,
        ),
        "Recall": recall_score(
            Y_test,
            y_pred,
            zero_division=0,
        ),
        "F1": f1_score(
            Y_test,
            y_pred,
            zero_division=0,
        ),
        "ROC-AUC": roc_auc_score(Y_test, y_score),
        "PR-AUC": average_precision_score(Y_test, y_score),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Predicted Churn": int(y_pred.sum()),
    })

In [20]:
test_results_df = (
    pd.DataFrame(test_results)
    .sort_values("PR-AUC", ascending=False)
    .reset_index(drop=True)
)

test_results_df

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP,Predicted Churn
0,Gradient Boosting,0.8700,0.732723,0.781609,0.501229,0.610778,0.869816,0.720927,1536,57,203,204,261
1,Random Forest,0.8605,0.699321,0.790909,0.427518,0.555024,0.860489,0.702151,1547,46,233,174,220
2,SVM RBF,0.8615,0.684400,0.853261,0.385749,0.531303,0.821338,0.675969,1566,27,250,157,184
3,Decision Tree,0.7795,0.768290,0.473602,0.749386,0.580400,0.832327,0.660747,1254,339,102,305,644


## Comparação CV versus teste

## Sobreposição dos churners identificados

In [22]:
prediction_comparison = pd.DataFrame(
    {
        "Actual": Y_test,
        **{
            model_name: pd.Series(
                predictions,
                index=Y_test.index,
            )
            for model_name, predictions
            in test_predictions.items()
        },
    },
    index=Y_test.index,
)

In [31]:
actual_churners = prediction_comparison[
    prediction_comparison["Actual"] == 1
].copy()


Quantos churners cada modelo identificou?

In [25]:
churn_hits = (
    actual_churners
    .drop(columns="Actual")
    .sum()
    .sort_values(ascending=False)
    .rename("Churners Identified")
    .to_frame()
)

display(churn_hits)

,Churners Identified
Decision Tree,305
Gradient Boosting,204
Random Forest,174
SVM RBF,157


Churners encontrados apenas pela Decision Tree

In [26]:
other_models = [
    "Gradient Boosting",
    "Random Forest",
    "SVM RBF",
]

tree_unique_churners = actual_churners[
    (actual_churners["Decision Tree"] == 1)
    & (actual_churners[other_models].sum(axis=1) == 0)
]

len(tree_unique_churners)

103

Churners identificados pela árvore e perdidos pelo Gradient Boosting

In [27]:
tree_hits_missed_by_gb = actual_churners[
    (actual_churners["Decision Tree"] == 1)
    & (actual_churners["Gradient Boosting"] == 0)
]

len(tree_hits_missed_by_gb)

105

Churners identificados por todos

In [28]:
model_columns = list(best_models.keys())

churners_identified_by_all = actual_churners[
    actual_churners[model_columns].eq(1).all(axis=1)
]

len(churners_identified_by_all)

147

Churners perdidos por todos

In [29]:
churners_missed_by_all = actual_churners[
    actual_churners[model_columns].eq(0).all(axis=1)
]

len(churners_missed_by_all)

97

In [30]:
pairwise_complementarity = []

for model_a in model_columns:
    for model_b in model_columns:
        if model_a >= model_b:
            continue

        a_hits_b_misses = (
            (actual_churners[model_a] == 1)
            & (actual_churners[model_b] == 0)
        ).sum()

        b_hits_a_misses = (
            (actual_churners[model_b] == 1)
            & (actual_churners[model_a] == 0)
        ).sum()

        both_hit = (
            (actual_churners[model_a] == 1)
            & (actual_churners[model_b] == 1)
        ).sum()

        pairwise_complementarity.append({
            "Model A": model_a,
            "Model B": model_b,
            "Both Identify": both_hit,
            "A Identifies / B Misses": a_hits_b_misses,
            "B Identifies / A Misses": b_hits_a_misses,
        })

pairwise_complementarity_df = pd.DataFrame(
    pairwise_complementarity
)

display(pairwise_complementarity_df)

,Model A,Model B,Both Identify,A Identifies / B Misses,B Identifies / A Misses
0,Gradient Boosting,Random Forest,172,32,2
1,Gradient Boosting,SVM RBF,156,48,1
2,Random Forest,SVM RBF,147,27,10
3,Decision Tree,Gradient Boosting,200,105,4
4,Decision Tree,Random Forest,174,131,0
5,Decision Tree,SVM RBF,155,150,2


               Decision Tree
          +----------------------+
          |                      |
          |    105 exclusivos    |
          |                      |
          |    +-----------+     |
          |    | GB        |     |
          |    | 200       |     |
          |    +-----------+     |
          |                      |
          +----------------------+

In [ ]:
ensemble = (
    (gb == 1)
    | (dt == 1)
).astype(int)